# W7 Lab — Research-Agent Workflow: Planner → Research → Writer → Editor

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ralbu85/stml_2026/blob/main/lectures/week07/W7_lab_research_agent.ipynb)

**Goal.** By the end of this lab you can assemble the semester's first half into one
system: a planner that decomposes a topic into steps (notes Ch. 7), an executor that
routes each step to specialist sub-agents (Ch. 6), a research agent that uses tools
in a loop (Ch. 3–4), and an editor that closes the reflection cycle (Ch. 5) — with
the final report scored by checklist and LLM judge (Ch. 5, target ≥ 4/5).

Why this week: the lecture closed the first half with planning; this lab is the
first half run as one machine, and the measured task is raising the report's quality
by refining the two prompts you own — the planner's and the editor's.

The path: setup → research tools → the sub-agents (planner ✍️, research, writer,
editor ✍️) → the executor → the measured report → ablations and cost accounting.

Adapted from DeepLearning.AI, *Agentic AI* by Andrew Ng — Module 5 graded lab /
final project (research agent). The workflow decomposition, agent roles, and
executor design follow the source lab; the setup is adapted for Google Colab and
the course API standard (`aisuite`, key pasted into the setup cell), and the
source's Tavily web-search tool (which needs its own API key) is replaced by
keyless arXiv and Wikipedia lookups with offline fallbacks.

*Runtime:* Google Colab, top-to-bottom, ~80 minutes. Cells marked ✍️ ask for your own writing — a fill-in or a written prediction.

## 1. Setup

### 1.1 Installation

`aisuite` exposes multiple providers behind one interface; `requests` (preinstalled
in Colab) carries the two research tools' HTTP calls.

In [ ]:
%pip install -q "aisuite[openai,anthropic]"

### 1.2 API key and model

Paste your key between the quotes (issuing steps: the API Setup guide on the course
site). The key is yours; do not share the notebook with the key still inside.

In [ ]:
import os

os.environ["OPENAI_API_KEY"] = "PASTE-YOUR-KEY-HERE"

MODEL = "openai:gpt-4o-mini"   # Anthropic accounts: MODEL = "anthropic:claude-haiku-4-5" and set ANTHROPIC_API_KEY instead

MAX_TOOL_TURNS = 12    # tool-call iterations allowed inside one research task (source lab value)
MAX_PLAN_STEPS = 4     # executed plan steps per workflow run, keeps cost and runtime bounded
HTTP_TIMEOUT = 15      # seconds per research-tool HTTP request

### 1.3 Client and helpers

`chat` and `ask` are the same helpers as in previous labs; both count calls and
tokens into module-level totals, read again in the exercises.

In [ ]:
import aisuite

client = aisuite.Client()

n_calls = 0
n_prompt_tokens = 0
n_completion_tokens = 0

def _track(response):
    """Response -> same response; updates call and token totals."""
    global n_calls, n_prompt_tokens, n_completion_tokens
    n_calls += 1
    usage = getattr(response, "usage", None)
    if usage is not None:
        n_prompt_tokens += usage.prompt_tokens
        n_completion_tokens += usage.completion_tokens
    return response

def chat(messages, temperature=0.0):
    """Message list -> assistant reply text."""
    response = _track(client.chat.completions.create(
        model=MODEL, messages=messages, temperature=temperature))
    return response.choices[0].message.content

def ask(prompt, system=None, temperature=0.0):
    """Single question (optional system instruction) -> reply text."""
    messages = ([{"role": "system", "content": system}] if system else [])
    messages.append({"role": "user", "content": prompt})
    return chat(messages, temperature=temperature)

### 1.4 Verification

In [ ]:
print(ask("Reply with exactly: ready"))

If the output is `ready`, key and billing work. Any error here is a setup problem,
not a code problem.

## 2. Research Tools

The research agent needs sources outside the model. Two keyless tools cover the
source lab's roles: the arXiv API for academic papers and the Wikipedia REST API
for encyclopedic background. Each function returns a list of result dicts; its
docstring and parameter descriptions become the tool schema that `aisuite` passes
to the model (notes Ch. 3: tool documentation is prompt engineering). When the network
is unavailable, both fall back to a small built-in corpus so the workflow still
runs end to end.

*Do:* run the cells and try one query per tool — these two functions are the system's only window on the world.


In [ ]:
import requests
import xml.etree.ElementTree as ET

# Wikimedia asks API clients for a descriptive User-Agent; generic ones get throttled.
HTTP_HEADERS = {"User-Agent": "STML-2026-lab/1.0"}

# MOCK: offline fallback corpus, used only when HTTP requests fail.
FALLBACK_NOTES = {
    "retrieval-augmented generation": (
        "Retrieval-augmented generation (RAG) couples a language model with a "
        "document index: at query time relevant passages are retrieved and placed "
        "into the model input, so answers can cite up-to-date, private sources. "
        "See Lewis et al., 2020, https://arxiv.org/abs/2005.11401"),
    "question answering": (
        "Question answering over private document collections combines a "
        "retriever over the collection with a reader model; evaluation uses "
        "answer accuracy and retrieval recall. "
        "See https://en.wikipedia.org/wiki/Question_answering"),
}

def _offline_lookup(query, source):
    """(query, source name) -> fallback result list from the built-in corpus."""
    matches = [{"title": key, "summary": text, "url": f"offline:{source}"}
               for key, text in FALLBACK_NOTES.items()
               if any(word in query.lower() for word in key.split())]
    return matches or [{"title": "no offline match", "summary": "", "url": f"offline:{source}"}]

def arxiv_search_tool(query: str, max_results: int = 3):
    """Searches arXiv for research papers matching the query.

    Args:
        query: Search keywords for research papers.
        max_results: Maximum number of papers to return.
    """
    url = ("https://export.arxiv.org/api/query?"
           f"search_query=all:{requests.utils.quote(query)}&start=0&max_results={max_results}")
    try:
        response = requests.get(url, headers=HTTP_HEADERS, timeout=HTTP_TIMEOUT)
        response.raise_for_status()
        ns = {"atom": "http://www.w3.org/2005/Atom"}
        root = ET.fromstring(response.content)
        results = []
        for entry in root.findall("atom:entry", ns):
            results.append({
                "title": entry.find("atom:title", ns).text.strip(),
                "published": entry.find("atom:published", ns).text[:10],
                "url": entry.find("atom:id", ns).text,
                "summary": entry.find("atom:summary", ns).text.strip()[:500],
            })
        return results or _offline_lookup(query, "arxiv")
    except Exception:
        return _offline_lookup(query, "arxiv")

def wikipedia_search_tool(query: str, sentences: int = 5):
    """Searches Wikipedia and returns a summary of the best-matching article.

    Args:
        query: Search keywords for the Wikipedia article.
        sentences: Approximate number of summary sentences to return.
    """
    try:
        search = requests.get(
            "https://en.wikipedia.org/w/api.php",
            params={"action": "opensearch", "search": query, "limit": 1, "format": "json"},
            headers=HTTP_HEADERS, timeout=HTTP_TIMEOUT).json()
        title = search[1][0]
        page = requests.get(
            f"https://en.wikipedia.org/api/rest_v1/page/summary/{requests.utils.quote(title)}",
            headers=HTTP_HEADERS, timeout=HTTP_TIMEOUT).json()
        summary = " ".join(page.get("extract", "").split(". ")[:sentences])
        return [{"title": page.get("title", title), "summary": summary,
                 "url": page.get("content_urls", {}).get("desktop", {}).get("page", "")}]
    except Exception:
        return _offline_lookup(query, "wikipedia")

A direct call to each tool, before any model is involved:

In [ ]:
for hit in arxiv_search_tool("retrieval augmented generation", max_results=2):
    print("ARXIV:", hit["title"][:70], "|", hit["url"])
for hit in wikipedia_search_tool("retrieval augmented generation"):
    print("WIKI :", hit["title"][:70], "|", hit["url"])

Each result carries a title, a summary, and a URL. The URL matters: the measured
task checks that the final report cites its sources, and a citation the code can
verify is a URL that appeared in a tool result.

## 3. Sub-Agents

The workflow divides labor among three specialists plus a planner. Each sub-agent
is one function: a model call with a role-specific instruction (notes Ch. 7:
plan-then-execute — the plan is produced once, then executed step by step).

### 3.1 Planner ✍️

`planner_agent` turns a topic into a plan: a Python list of step strings. Write
`PLANNER_PROMPT` (placeholder `{topic}` required). It must instruct the model to:

- return ONLY a valid Python list of strings — no prose, no code fences;
- make each step atomic and executable by one of the three agents below
  (research / writer / editor) — no file handling, no package installation;
- end with a final step that produces the complete research report in Markdown;
- keep the plan to at most 4 steps — the executor (Section 4) runs only the first
  `MAX_PLAN_STEPS` (= 4) and silently drops the rest.

In [ ]:
### FILL IN (START) ###
PLANNER_PROMPT = (
    "{topic}"
)
### FILL IN (END) ###

In [ ]:
import ast
import re

def strip_code_fences(text):
    """Model output -> same text with surrounding Markdown code fences removed."""
    return re.sub(r"^```(?:python|json)?\s*|\s*```$", "", text.strip())

def planner_agent(topic, temperature=1.0):
    """Topic -> list of plan-step strings; raises ValueError on malformed output."""
    raw = ask(PLANNER_PROMPT.format(topic=topic), temperature=temperature)
    try:
        # literal_eval parses the list without executing code (the source lab used eval)
        steps = ast.literal_eval(strip_code_fences(raw))
    except (ValueError, SyntaxError) as exc:
        raise ValueError(f"planner output is not a Python list: {raw[:120]!r}") from exc
    if not (isinstance(steps, list) and all(isinstance(s, str) for s in steps)):
        raise ValueError(f"planner output is not a list of strings: {raw[:120]!r}")
    return steps

The plan for this lab's fixed topic. Temperature 1.0 follows the source lab:
planning benefits from sampled variety, and the executor tolerates different
valid plans.

In [ ]:
TOPIC = ("Retrieval-augmented generation for question answering "
         "over private document collections")

plan_steps = planner_agent(TOPIC)
for i, step in enumerate(plan_steps, 1):
    print(f"{i}. {step}")

A `ValueError` here means `PLANNER_PROMPT` does not yet force the list-only output
format — the same format-control problem as W1, now load-bearing: every later cell
consumes this list.


### 3.2 Research agent

Near-verbatim from the source lab: the model receives the task and the two tool
functions, and calls them as it sees fit for up to `MAX_TOOL_TURNS` iterations
(`aisuite` runs the tool loop internally, as in W4).

In [ ]:
from datetime import date

def research_agent(task):
    """Research task -> text answer grounded in tool results."""
    print("== Research Agent ==")
    prompt = (
        "You are a research assistant with access to these tools:\n"
        "- arxiv_search_tool: finds academic papers\n"
        "- wikipedia_search_tool: retrieves encyclopedic summaries\n\n"
        "Use the tools to gather material, then answer the task. For every source "
        "you use, include its title and URL in your answer.\n\n"
        f"Task:\n{task}\n\nToday is {date.today().isoformat()}."
    )
    response = _track(client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        tools=[arxiv_search_tool, wikipedia_search_tool],
        max_turns=MAX_TOOL_TURNS,
    ))
    return response.choices[0].message.content or ""

### 3.3 Writer agent

Verbatim role from the source lab; temperature 1.0 for drafting.

In [ ]:
WRITER_SYSTEM = ("You are a writing agent specialized in generating "
                 "well-structured academic or technical content.")

def writer_agent(task):
    """Writing task -> drafted or revised text."""
    print("== Writer Agent ==")
    return ask(task, system=WRITER_SYSTEM, temperature=1.0)

### 3.4 Editor ✍️

The editor closes the reflection loop (notes Ch. 5: reflection). Write
`EDITOR_SYSTEM`. It must instruct the model that:

- its role is an editor for research reports;
- when asked to critique or review, it returns numbered, actionable feedback
  judged against structure (clear Markdown sections), grounding (claims tied to
  cited sources with URLs), and length (concise, no padding);
- when asked to revise, it returns the complete revised Markdown report itself,
  not commentary about it.

In [ ]:
### FILL IN (START) ###
EDITOR_SYSTEM = (
    ""
)
### FILL IN (END) ###

def editor_agent(task):
    """Editorial task -> critique text, or the revised report when revision is asked."""
    print("== Editor Agent ==")
    return ask(task, system=EDITOR_SYSTEM or None, temperature=0.7)

## 4. Executor

The executor walks the plan. For each step it asks a model which agent should act
and with what task (raw JSON, one routing decision per step — notes Ch. 1 §1.4: routing),
rebuilds the context from previous outputs, and dispatches through the registry.
The procedure per step:

1. **Route** — a model call maps the step to `{"agent": ..., "task": ...}`.
2. **Enrich** — the task is prefixed with all previous steps' outputs.
3. **Dispatch** — the registry calls the chosen sub-agent function.
4. **Record** — `(step, agent, output)` is appended to the history.

*Do:* run the cell and read the four-step procedure — route, enrich, dispatch, record.


In [ ]:
import json

agent_registry = {
    "research_agent": research_agent,
    "writer_agent": writer_agent,
    "editor_agent": editor_agent,
}

ROUTING_PROMPT = """You are an execution manager for a multi-agent research team.

Given the following instruction, identify which agent should perform it and extract the clean task.

Return only a valid JSON object with two keys:
- "agent": one of ["research_agent", "editor_agent", "writer_agent"]
- "task": a string with the instruction that the agent should follow

Only respond with a valid JSON object. Do not include explanations or markdown formatting.

Instruction: "{step}"
"""

def executor_agent(plan_steps):
    """Plan-step list -> history of (step, agent name, output) tuples."""
    history = []
    for step in plan_steps[:MAX_PLAN_STEPS]:
        decision = json.loads(strip_code_fences(
            ask(ROUTING_PROMPT.format(step=step))))
        agent_name, task = decision["agent"], decision["task"]

        context = "\n".join(
            f"Step {j+1} executed by {a}:\n{r}"
            for j, (s, a, r) in enumerate(history))
        enriched_task = (f"You are {agent_name}.\n\n"
                         f"Here is the context of what has been done so far:\n{context}\n\n"
                         f"Your next task is:\n{task}")

        print(f"\n-> {agent_name}: {task[:80]}")
        if agent_name in agent_registry:
            output = agent_registry[agent_name](enriched_task)
        else:
            output = f"unknown agent: {agent_name}"
        history.append((step, agent_name, output))
    return history

The full workflow, end to end:

In [ ]:
calls_before = n_calls
history = executor_agent(plan_steps)
workflow_calls = n_calls - calls_before

final_report = history[-1][2]
print(f"\nsteps executed: {len(history)}   model calls: {workflow_calls}")

In [ ]:
from IPython.display import Markdown, display

display(Markdown(final_report))

Read the report against the plan: each plan step should be visible in the result —
researched sources cited by URL, a drafted structure, and the editor's revisions.
Where a step left no trace, the routing decision or the sub-agent prompt lost it;
the history tuples locate which. A `JSONDecodeError` inside the executor means the
router replied with prose instead of the JSON object — re-run the step; if it
repeats, the plan step's wording is confusing the router.


## 5. Measured Task — Report Quality ✍️ (core)

The claim "the workflow produced a good report" becomes a measurement with two
layers, scored on the final history entry:

| layer | checks |
|---|---|
| programmatic | ≥ 3 Markdown headings · 300–1500 words · ≥ 2 URLs cited · final step by writer or editor |
| LLM judge (0–5) | grounding (claims tied to the cited sources) · coverage of the topic · structure |

**Target: all 4 programmatic checks pass and judge score ≥ 4.** Most of the
remaining time belongs here: iterate on `PLANNER_PROMPT` and `EDITOR_SYSTEM`
(one change at a time), re-run Section 4, re-score.

In [ ]:
#@title Report checks and judge — run as-is (criteria described above) { display-mode: "form" }
JUDGE_PROMPT = """You are grading a research report against its sources.

Criteria: grounding (claims tied to the listed sources), coverage of the topic's
main aspects, and structure (clear sections in logical order).

Topic: {topic}

Sources gathered during research:
{sources}

Report:
{report}

Reply with exactly one line: SCORE: <integer 0-5>, then one sentence of justification."""

def judge_report(report, history, topic=None):
    """(report, workflow history) -> integer judge score 0-5."""
    sources = "\n".join(out[:300] for _, agent, out in history
                        if agent == "research_agent") or "(none recorded)"
    reply = ask(JUDGE_PROMPT.format(topic=topic or TOPIC, sources=sources, report=report))
    match = re.search(r"SCORE:\s*(\d)", reply)
    if match is None:
        raise ValueError(f"judge reply has no SCORE line: {reply[:120]!r}")
    print(reply.strip())
    return int(match.group(1))

def check_report(report, history):
    """(report, history) -> dict of 4 named boolean checks."""
    headings = report.count("\n#") + (1 if report.lstrip().startswith("#") else 0)
    words = len(report.split())
    return {
        "three_headings":   headings >= 3,
        "length_in_bounds": 300 <= words <= 1500,
        "two_urls_cited":   report.count("http") >= 2,
        "final_step_role":  history[-1][1] in ("writer_agent", "editor_agent"),
    }

In [ ]:
JUDGE_TARGET = 4

report_checks = check_report(final_report, history)
for name, ok in report_checks.items():
    print(f"{'PASS' if ok else 'FAIL':4}  {name}")
judge_score = judge_report(final_report, history)
print(f"\njudge: {judge_score}/5   "
      + ("TARGET REACHED" if all(report_checks.values()) and judge_score >= JUDGE_TARGET
         else "KEEP ITERATING"))

A failing `two_urls_cited` usually traces to the planner (no explicit research
steps) or to the editor dropping citations during revision — the fill-in prompts
control both. A low judge score with passing programmatic checks means the report
is well-formed but weakly grounded; tighten the editor's grounding requirement. A
`final_step_role` failure with a long plan means the report step fell past
`MAX_PLAN_STEPS` and was truncated — shorten the plan.


## 6. Exercises ✍️

Protocol: write your prediction down first, run the cell, compare.

### 6.1 Editor ablation

The variant below executes the same plan minus every step the router sends to the
editor. Prediction: how do the judge score and the call count change?
If the two step counts printed by the cell match (nothing was removed), your plan
words its editing step differently — adjust the filter regex first.

In [ ]:
calls_before = n_calls
plan_no_editor = [s for s in plan_steps
                  if not re.search(r"review|revise|edit|feedback|critique", s,
                                   re.IGNORECASE)]
print(f"plan steps: {len(plan_steps)} -> {len(plan_no_editor)} after removing editor steps")
history_no_editor = executor_agent(plan_no_editor)
ablation_calls = n_calls - calls_before

no_editor_report = history_no_editor[-1][2]
no_editor_score = judge_report(no_editor_report, history_no_editor)
print(f"\nfull workflow : judge {judge_score}/5, {workflow_calls} calls")
print(f"editor removed: judge {no_editor_score}/5, {ablation_calls} calls")

The editor's contribution is bought with extra calls; whether the score drop
justifies the savings is the same cost–quality trade every stage of a workflow
faces (→ notes Ch. 12).


### 6.2 Plan length

The variant below executes only the first two plan steps. Prediction: which
programmatic checks fail first when the plan is cut short?

In [ ]:
history_short = executor_agent(plan_steps[:2])
short_report = history_short[-1][2]
for name, ok in check_report(short_report, history_short).items():
    print(f"{'PASS' if ok else 'FAIL':4}  {name}")

With research steps but no drafting step, the last output is source notes, not a
report — the plan's final write-the-report step is what makes the history's tail a
deliverable. A plan is not decoration; the executor produces exactly what the plan
asks for and nothing more.

### 6.3 Cost accounting

The whole session, priced at gpt-4o-mini list prices.

In [ ]:
PRICE_PER_M_PROMPT = 0.15        # USD per 1M input tokens, gpt-4o-mini
PRICE_PER_M_COMPLETION = 0.60    # USD per 1M output tokens, gpt-4o-mini

cost = (n_prompt_tokens * PRICE_PER_M_PROMPT
        + n_completion_tokens * PRICE_PER_M_COMPLETION) / 1_000_000
print(f"calls: {n_calls}   prompt tokens: {n_prompt_tokens}   "
      f"completion tokens: {n_completion_tokens}")
print(f"estimated session cost: ${cost:.4f}")

Every plan step costs a routing call plus the sub-agent's calls, and the research
agent's tool loop multiplies further. Multi-step workflows are where call counts
stop being negligible (→ notes Ch. 12).


## 7. Completion Check

All rows must read `PASS` before submission; grading checks these same structural
facts, never prose quality.

In [ ]:
completion = {
    "PLANNER_PROMPT written (>= 80 chars, keeps {topic})":
        len(PLANNER_PROMPT.strip()) >= 80 and "{topic}" in PLANNER_PROMPT,
    "EDITOR_SYSTEM written (>= 40 chars)":
        len(EDITOR_SYSTEM.strip()) >= 40,
    "workflow executed (history recorded)":
        len(history) >= 1,
    "programmatic report checks all pass":
        all(report_checks.values()),
    "judge target reached (>= 4/5)":
        judge_score >= JUDGE_TARGET,
}
for item, ok in completion.items():
    print(f"{'PASS' if ok else 'FAIL':4}  {item}")
print("\nLAB COMPLETE" if all(completion.values()) else "\nNOT COMPLETE YET")

---

This closes the first-half lab track: W1–2 built prompting control, W3 tools,
W4 the agent loop, W5 the reflection-and-evaluation loop, W6 multi-agent handoffs,
and this lab composed them under a plan. The W8 midterm
covers weeks 1–7; labs resume in W9 with the retriever for the final project.
Reference answers for this lab and the homework:
`labs/checkpoints/week07/solution.py`, published after the homework deadline.